# Día 3 · Recalibración del juez de relevancia

La auditoría humana encontró que el juez v1 fue demasiado estricto. Este notebook conserva exactamente los rankings normal/HyDE y reevalúa únicamente la relevancia con el criterio humano: un fragmento es relevante si aporta evidencia directa, parcial o contexto útil.

In [ ]:
!git clone -q https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git
%cd proyecto_ActividadGrado-Riesgos
!git checkout -q dia-03-experimento-hyde
!pip -q install pandas openpyxl openai

## 1. Subir el ZIP anterior

Selecciona `resultados_hyde_dia_03.zip`. No se vuelve a ejecutar BGE-M3 ni se regeneran los documentos HyDE.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
input_dir = Path('/content/hyde_v1')
input_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(input_dir)
print('Resultados anteriores cargados:', input_dir)

## 2. Cargar la clave temporal

In [ ]:
import os
from getpass import getpass
os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

## 3. Reevaluar relevancia

Son aproximadamente 20 llamadas. El proceso conserva un checkpoint y normalmente tarda menos que el experimento inicial.

In [ ]:
!python src/retrieval/rejudge_hyde_results.py \
  --results /content/hyde_v1/retrieval_results_evaluated.csv \
  --questions data/evaluation/gold_questions.csv \
  --output-dir /content/resultados_hyde_juez_v2

## 4. Revisar y descargar

In [ ]:
import pandas as pd, shutil
from IPython.display import display
out = Path('/content/resultados_hyde_juez_v2')
display(pd.read_csv(out / 'metrics_by_method_v2.csv'))
result_zip = shutil.make_archive('/content/resultados_hyde_juez_v2', 'zip', out)
files.download(result_zip)